# RAG Under The Hood: ML4NLP Assignment 4 Assistant

We will build a RAG pipeline to answer questions about assignment 4 using LangChain and Gemini. We will cover:
1.  **Indexing:** Loading a website, splitting the text into chunks, and storing them in a vector database.
2.  **Visualization:** Inspecting how documents are split and what vector embeddings look like.
3.  **Retrieval & Generation:** Setting up the chain to search for answers and generate a response using Google's Gemini model.

## 1. Setup
First, we need to install the necessary libraries.

In [ ]:
# !pip install langchain langchain-community langchain-google-genai langchain-chroma bs4 python-dotenv

### Environment Setup

To run this notebook, you need to configure your API keys. We use **Google Gemini** and **LangSmith** for tracing.

Create a file named `.env` in the same folder as this notebook and paste the following content:

```bash
GOOGLE_API_KEY=your_key_here
LANGSMITH_TRACING_V2=true
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_API_KEY=your_key_here
LANGSMITH_PROJECT=your_project_name_here
```

**Get your keys here (free to use):**
*   **Google API Key:** [https://aistudio.google.com/api-keys](https://aistudio.google.com/api-keys)
*   **LangSmith API Key:** [https://smith.langchain.com](https://smith.langchain.com)

In [ ]:
import os
import bs4
from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain_community.document_loaders import WebBaseLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv()

# Setup Gemini API Key and LangSmith tracing
api_key = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
os.environ["USER_AGENT"] = "my-test-agent/1.0"

## 2. Indexing: Loading and Splitting
We will load the content of **Assignment 4** from the course website.

Large documents cannot be fed into an LLM all at once. We must **split** (chunk) them into smaller pieces.
*   **Chunk Size:** How large each piece is (1000 characters).
*   **Chunk Overlap:** How much text is repeated between chunks (200 characters). 

In [41]:
# Load contents of assignment 4
loader = WebBaseLoader("https://dsai-nlp.github.io/courses/dat450/assignment4")
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 19942


In [42]:
print(docs[0].page_content[:500])

       DAT450/DIT247: Programming Assignment 4: Supervised Fine-Tuning (SFT) with LoRA | NLP@DSAI                   NLP@DSAI  Toggle navigation        NLP@DSAI   Members   Events   Publications   News   Courses          DAT450/DIT247: Programming Assignment 4: Supervised Fine-Tuning (SFT) with LoRA In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM (preferably OLMo-2 1B) on Alpaca, a dataset of 52k instructions generated by OpenAI’s text-davinci-003 engine. You


In [43]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split assignment info into {len(all_splits)} sub-documents.")

Split assignment info into 32 sub-documents.


In [44]:
# Print details for the first 3 splits
for i, doc in enumerate(all_splits[:3]):
    print(f"--- Split {i} ---")
    print(f"Length: {len(doc.page_content)} characters")
    print(f"Metadata: {doc.metadata}")
    print("-" * 20)
    print(doc.page_content[:150] + "...")
    print("\n")

--- Split 0 ---
Length: 991 characters
Metadata: {'source': 'https://dsai-nlp.github.io/courses/dat450/assignment4', 'title': 'DAT450/DIT247: Programming Assignment 4: Supervised Fine-Tuning (SFT) with LoRA | NLP@DSAI', 'description': 'NLP@DSAI is a constellation of researchers who carry out foundational or applied research in natural language processing (NLP), or are interested in NLP techniques generally. ', 'language': 'en', 'start_index': 7}
--------------------
DAT450/DIT247: Programming Assignment 4: Supervised Fine-Tuning (SFT) with LoRA | NLP@DSAI                   NLP@DSAI  Toggle navigation        NLP@DS...


--- Split 1 ---
Length: 994 characters
Metadata: {'source': 'https://dsai-nlp.github.io/courses/dat450/assignment4', 'title': 'DAT450/DIT247: Programming Assignment 4: Supervised Fine-Tuning (SFT) with LoRA | NLP@DSAI', 'description': 'NLP@DSAI is a constellation of researchers who carry out foundational or applied research in natural language processing (NLP), or are in

In [45]:
def visualize_overlap(splits, index=0):
    """Visualizes the overlap between chunk [index] and chunk [index+1]"""
    if index + 1 >= len(splits):
        print("Index out of bounds")
        return

    chunk_1 = splits[index]
    chunk_2 = splits[index + 1]
    
    overlap_size = 0
    for i in range(1, len(chunk_2.page_content)):
        snippet = chunk_2.page_content[:i]
        if chunk_1.page_content.endswith(snippet):
            overlap_size = len(snippet)
        else:
            if i > 1000: break 
            
    print(f"Comparing Split {index} and Split {index+1}")
    print(f"Calculated Overlap: ~{overlap_size} characters")
    print("-" * 40)
    print(f"END OF SPLIT {index}:   ...{chunk_1.page_content[-200:]!r}")
    print(f"START OF SPLIT {index+1}: {chunk_2.page_content[:200]!r}")
    print("-" * 40)

visualize_overlap(all_splits, index=1)

Comparing Split 1 and Split 2
Calculated Overlap: ~192 characters
----------------------------------------
END OF SPLIT 1:   ...'already designed for the code) This is a pure programming assignment and you do not have to write a technical report or explain details of your solution: there will be a separate individual assignment'
START OF SPLIT 2: 'designed for the code) This is a pure programming assignment and you do not have to write a technical report or explain details of your solution: there will be a separate individual assignment where y'
----------------------------------------


## 3. Embeddings and Vector Store
Now we turn text into numbers (**Embeddings**).
*   **Embedding Model:** `text-embedding-004` (Google). Translates text into a vector of 768 numbers.
*   **Vector Store:** `Chroma`. A database designed to store and search these vectors efficiently.

In [46]:
# Initialize Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    google_api_key=api_key
)

# Initialize the Vector Store with Gemini
vector_store = Chroma(
    collection_name="assignment4_rag",
    embedding_function=embeddings,
    # persist_directory="./chroma_db"
)

# Add splits
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Success! Added {len(document_ids)} chunks using Gemini Embeddings.")
print(f"Sample ID: {document_ids[0]}")

Success! Added 32 chunks using Gemini Embeddings.
Sample ID: 5fa69963-006d-4a50-b56b-72e1a1576c78


In [47]:
# View vector embeddings
data = vector_store.get(include=['embeddings', 'documents', 'metadatas'])
first_embedding = data['embeddings'][0]

print(f"Total Vectors stored: {len(data['embeddings'])}")
print(f"Dimension of each vector: {len(first_embedding)}") 
print("-" * 30)
print(f"Sample of first 10 numbers in Vector 0:\n{first_embedding[:10]}")
print("...")

Total Vectors stored: 64
Dimension of each vector: 768
------------------------------
Sample of first 10 numbers in Vector 0:
[-0.00101495  0.02211403 -0.02108049  0.01573845  0.04285909  0.02871293
  0.04483035  0.0560062   0.05765704 -0.00099739]
...


## 4. Retrieval and Generation (RAG)
Now we build the actual application.
1.  **Retriever:** Finds the most relevant chunks using MMR (Maximal Marginal Relevance) to ensure diversity.
2.  **LLM:** The chat model (`gemini-2.5-flash`) that will read the chunks and answer.
3.  **Chain:** Connects the Retriever -> Prompt -> LLM.

In [48]:
# Setup the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1
)

# Setup the retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7}
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [49]:
# Define the Prompt
template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Build RAG Chain
# Source Data -> Prompt -> Model -> String Output
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() # Cleans up the output to be only string
)

In [50]:
# Test the Chain
query = "What is the deadline for assignment 4?"
# query = "What dataset is used in the assignment?"

print(f"Question: {query}")
print("-" * 20)

response = rag_chain.invoke(query)

print(response)

Question: What is the deadline for assignment 4?
--------------------
The deadline for assignment 4 is December 1. You need to submit Python files containing your solution to the programming tasks. Additionally, a text file with the outputs printed by your Python program should be submitted.
